In [0]:
# Databricks notebook source
# ETL - Squad 3 - ecommerce_clientes
# Fluxo: Raw CSV -> Bronze Delta -> Silver Delta -> Gold Delta/SQL Server

In [0]:
%run "../config/00_config"

In [0]:
%run "../config/tables/ecommerce_clientes_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# definir caminhos Silver e Gold (mudar para o config)

SILVER_TABLE = "ecommerce_clientes"
SILVER_PATH = f"{SILVER_BASE_PATH}{SILVER_TABLE}"
SILVER_WRITE_MODE = "overwrite"

# Gold ainda será definida depois que o KPI for combinado com o time
GOLD_TABLE = "ecommerce_clientes_kpi_a_definir"
GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"
GOLD_WRITE_MODE = "overwrite"

print("SOURCE_PATH:", SOURCE_PATH)
print("BRONZE_PATH:", BRONZE_PATH)
print("SILVER_PATH:", SILVER_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("PARTITION_DATE_COLUMN:", PARTITION_DATE_COLUMN)

In [0]:
# montar opções ADLS

adls_options = build_adls_options(
    ADLS_STORAGE_ACCOUNT_NAME,
    ADLS_CLIENT_ID,
    ADLS_TENANT_ID,
    ADLS_CLIENT_SECRET
)

print("Opções ADLS configuradas com OAuth/Service Principal.")

In [0]:

# ler CSV da Raw

df_source = read_source_csv(
    spark=spark,
    source_path=SOURCE_PATH,
    adls_options=adls_options,
    csv_options=CSV_OPTIONS
)

df_source.printSchema()
display(df_source.limit(10))

In [0]:
# contar origem

total_source = df_source.count()

print(f"Total de registros lidos da Raw: {total_source}")

In [0]:
# validar conversão da data de particionamento

from pyspark.sql.functions import col, to_timestamp, count, when

df_test_date = df_source.withColumn(
    "dt_cadastro_convertida",
    to_timestamp(col(PARTITION_DATE_COLUMN))
)

display(
    df_test_date.select(
        count("*").alias("total_linhas"),
        count(when(col(PARTITION_DATE_COLUMN).isNull(), True)).alias("dt_cadastro_nula_origem"),
        count(
            when(
                col(PARTITION_DATE_COLUMN).isNotNull() &
                col("dt_cadastro_convertida").isNull(),
                True
            )
        ).alias("falhas_conversao")
    )
)

In [0]:

# criar DataFrame Bronze

from pyspark.sql.functions import current_timestamp, lit, year, month

df_bronze = (
    df_source
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", lit(SOURCE_PATH))
    .withColumn("_partition_date", to_timestamp(col(PARTITION_DATE_COLUMN)))
    .withColumn("ano", year(col("_partition_date")))
    .withColumn("mes", month(col("_partition_date")))
    .drop("_partition_date")
)

In [0]:
# visualizar Bronze antes de gravar

display(
    df_bronze
    .select(
        "id_cliente",
        "dt_cadastro",
        "ano",
        "mes",
        "bronze_ingested_at",
        "bronze_source_file"
    )
    .limit(20)
)

In [0]:
# validar particionamento Bronze
display(
    df_bronze.select(
        count("*").alias("total_linhas"),
        count(when(col("ano").isNull(), True)).alias("ano_nulo"),
        count(when(col("mes").isNull(), True)).alias("mes_nulo")
    )
)

In [0]:
# gravar Bronze Delta

(
    df_bronze
    .write
    .format("delta")
    .options(**adls_options)
    .mode(BRONZE_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(BRONZE_PATH)
)

print(f"Dados gravados com sucesso na Bronze: {BRONZE_PATH}")

In [0]:
# ler Bronze gravada

df_bronze_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(BRONZE_PATH)
)

df_bronze_saved.printSchema()

total_bronze = df_bronze_saved.count()

print(f"Total de registros na Bronze: {total_bronze}")

display(df_bronze_saved.limit(10))

In [0]:
# validar origem x Bronze

print(f"Total origem Raw: {total_source}")
print(f"Total gravado Bronze: {total_bronze}")

if total_source == total_bronze:
    print("Validação Bronze OK: quantidade de registros da Raw e da Bronze é igual.")
else:
    print("Atenção: quantidade de registros diferente entre Raw e Bronze.")

In [0]:
from pyspark.sql.functions import col, trim, lower, to_timestamp, current_timestamp

df_silver = (
    df_bronze_saved
    .select(
        col("id_cliente").cast("int").alias("id_cliente"),
        col("uuid_cliente").cast("string").alias("uuid_cliente"),
        trim(col("nome")).alias("nome"),
        trim(col("sobrenome")).alias("sobrenome"),
        lower(trim(col("email"))).alias("email"),
        col("senha_hash").cast("string").alias("senha_hash"),
        to_timestamp(col("dt_cadastro")).alias("dt_cadastro"),
        to_timestamp(col("dt_ultima_atualizacao")).alias("dt_ultima_atualizacao"),

        # metadados vindos da Bronze
        col("bronze_ingested_at").cast("timestamp").alias("bronze_ingested_at"),
        col("bronze_source_file").cast("string").alias("bronze_source_file"),

        # metadado da Silver
        current_timestamp().alias("silver_processed_at"),

        # partições
        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes")
    )
)

In [0]:
# tratar possíveis duplicidades por id_cliente
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc_nulls_last

window_cliente = (
    Window
    .partitionBy("id_cliente")
    .orderBy(
        desc_nulls_last("dt_ultima_atualizacao"),
        desc_nulls_last("dt_cadastro")
    )
)

df_silver_dedup = (
    df_silver
    .withColumn("_row_number", row_number().over(window_cliente))
    .filter(col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
# validar Silver antes de gravar
display(
    df_silver_dedup.select(
        count("*").alias("total_linhas_silver"),
        count(when(col("id_cliente").isNull(), True)).alias("id_cliente_nulo"),
        count(when(col("dt_cadastro").isNull(), True)).alias("dt_cadastro_nulo"),
        count(when(col("ano").isNull(), True)).alias("ano_nulo"),
        count(when(col("mes").isNull(), True)).alias("mes_nulo")
    )
)

display(df_silver_dedup.limit(10))

In [0]:
# gravar Silver Delta

(
    df_silver_dedup
    .write
    .format("delta")
    .options(**adls_options)
    .mode(SILVER_WRITE_MODE)
    .partitionBy("ano", "mes")
    .save(SILVER_PATH)
)

print(f"Dados gravados com sucesso na Silver: {SILVER_PATH}")

In [0]:
# ler Silver gravada

df_silver_saved = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_PATH)
)

df_silver_saved.printSchema()

total_silver = df_silver_saved.count()

print(f"Total de registros na Silver: {total_silver}")

display(df_silver_saved.limit(10))

In [0]:
# TODO: definir com o time/tech lead qual KPI será criado para ecommerce_clientes.
#
# Exemplo possível, ainda NÃO confirmado:
# - clientes cadastrados por ano e mês
#
# Possível saída:
# GOLD_TABLE = "ecommerce_clientes_kpi_cadastros_mensais"
# GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_TABLE}"
#
# df_gold = (
#     df_silver_saved
#     .groupBy("ano", "mes")
#     .agg(
#         count("*").alias("total_clientes_cadastrados")
#     )
#     .withColumn("gold_processed_at", current_timestamp())
# )
#
# display(df_gold)

In [0]:
# TODO: gravar a tabela Gold no SQL Server depois que o KPI for definido.
#
# Exemplo futuro:
# write_sql_table(
#     df=df_gold,
#     table_name="ecommerce_clientes_kpi_cadastros_mensais",
#     mode="overwrite"
# )

In [0]:
print("Resumo da execução ETL - ecommerce_clientes")
print(f"Origem Raw: {SOURCE_PATH}")
print(f"Destino Bronze: {BRONZE_PATH}")
print(f"Destino Silver: {SILVER_PATH}")
print(f"Destino Gold: {GOLD_PATH} - ainda não implementado")
print(f"Total origem Raw: {total_source}")
print(f"Total Bronze: {total_bronze}")
print(f"Total Silver: {total_silver}")
print("Particionamento: ano, mes")

if total_source == total_bronze:
    print("Status Bronze: SUCESSO")
else:
    print("Status Bronze: ATENÇÃO")

print("Status Silver: SUCESSO")
print("Status Gold: PENDENTE - KPI ainda não definido")
print("Status SQL Server: PENDENTE - depende da Gold")